# Thesis Notebook

## Install dependencies

In [ ]:
!pip install --upgrade pip

In [ ]:
!pip install --upgrade transformers datasets peft bitsandbytes accelerate evaluate seqeval fsspec huggingface_hub

In [ ]:
# check gpu driver
!nvidia-smi

## Training

### Training BERT with LoRA for Question Answering

#### Main Experiment Function

In [ ]:
# === qa_lora_squad.py ===
import os
import evaluate
import numpy as np
import torch
import gc

from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
    pipeline,
    set_seed,
)
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
)


def run_lora_experiment(
    seed: int,
    base_model: str = "bert-base-uncased",
    base_dataset: str = "squad",
    output_dir: str = "./results",
    logging_dir: str = "./logs",
    num_epochs: int = 3,
    batch_size: int = 32,
    learning_rate: float = 5e-5,
    device: int = 0,  # GPU device index, -1 for CPU
):
    """
    Run a full train + evaluate pipeline with LoRA on a QA task for a given seed.

    Parameters:
    - seed: Random seed for reproducibility.
    - base_model: Hugging Face model identifier.
    - base_dataset: Hugging Face dataset identifier.
    - output_dir: Root directory for saving model & logs.
    - logging_dir: Directory for training logs.
    - num_epochs: Number of training epochs.
    - batch_size: Training batch size.
    - learning_rate: Initial learning rate.
    - device: GPU device index or -1 for CPU.
    """
    # Set seed
    set_seed(seed)

    # Prepare tokenizer
    tokenizer = BertTokenizerFast.from_pretrained(base_model)

    # Reset GPU memory stats if using GPU
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()  # Reset peak memory stats if using GPU
    if device >= 0:
        torch.cuda.set_device(device)
    else:
        device = -1

    # Preprocessing function
    def preprocess_fn(ex):
        tok = tokenizer(
            ex["question"], ex["context"],
            truncation="only_second",
            max_length=384,
            stride=128,
            return_overflowing_tokens=False,
            return_offsets_mapping=True,
            padding="max_length",
        )
        offsets = tok.pop("offset_mapping")
        start_char = ex["answers"]["answer_start"][0]
        end_char = start_char + len(ex["answers"]["text"][0])
        start_idx = end_idx = 0
        for i, (s, e) in enumerate(offsets):
            if s <= start_char < e:
                start_idx = i
            if s < end_char <= e:
                end_idx = i
                break
        tok["start_positions"] = start_idx
        tok["end_positions"] = end_idx
        return tok

    # Load and preprocess dataset
    raw = load_dataset(base_dataset)
    train_ds = raw["train"].map(preprocess_fn, batched=False)
    val_ds = raw["validation"].map(preprocess_fn, batched=False)
    train_ds.set_format(type="torch", columns=["input_ids","attention_mask","start_positions","end_positions"])
    val_ds.set_format(type="torch", columns=["input_ids","attention_mask","start_positions","end_positions"])

    # Model + LoRA setup
    model = BertForQuestionAnswering.from_pretrained(base_model)
    lora_cfg = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["query","value"],
        lora_dropout=0.1,
        bias="none",
        task_type="QUESTION_ANS",
        use_rslora=False,
    )
    model = get_peft_model(model, lora_cfg)

    # Training arguments
    train_args = TrainingArguments(
        output_dir=os.path.join(output_dir, f"seed_{seed}"),
        per_device_train_batch_size=batch_size,
        num_train_epochs=num_epochs,
        learning_rate=learning_rate,
        fp16=torch.cuda.is_available(),
        logging_dir=os.path.join(logging_dir, f"seed_{seed}"),
        logging_strategy="steps",
        logging_steps=500,
        save_strategy="epoch",
        eval_strategy="epoch",
        report_to="none",
        label_names=["start_positions", "end_positions"],
    )
    trainer = Trainer(
        model=model,
        args=train_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=default_data_collator,
    )

    # Training
    print(f"Training with seed {seed}...")
    trainer.train()

    # Check peak memory
    peak_mem = None
    if torch.cuda.is_available():
        peak_mem = torch.cuda.max_memory_allocated() / (1024**3)  # in GB
        print(f"Peak CUDA memory (GB): {peak_mem:.2f}")

    # Save model and tokenizer
    trainer.save_model(os.path.join(output_dir, f"lora_squad_seed_{seed}"))
    tokenizer.save_pretrained(os.path.join(output_dir, f"lora_squad_seed_{seed}"))

    # Evaluate
    print("Evaluating model...")
    # Load LoRA-wrapped model for inference
    base = BertForQuestionAnswering.from_pretrained(base_model)
    infer_model = PeftModel.from_pretrained(base, os.path.join(output_dir, f"seed_{seed}"))
    qa_pipe = pipeline("question-answering", model=infer_model, tokenizer=tokenizer, device=device)
    dataset = load_dataset(base_dataset, split="validation")
    predictions, references = [], []
    for item in dataset:
        out = qa_pipe(question=item["question"], context=item["context"])
        predictions.append({"id": item["id"], "prediction_text": out.get("answer", "")})
        references.append({"id": item["id"], "answers": item["answers"]})
    metric = evaluate.load(base_dataset)
    results = metric.compute(predictions=predictions, references=references)
    print(f"Results (EM / F1): {results['exact_match']:.2f} / {results['f1']:.2f}")

    # Cleanup
    torch.cuda.empty_cache()
    del model, infer_model, trainer
    gc.collect()

    # print into text file
    with open(os.path.join(output_dir, f"results_seed_{seed}.txt"), "w") as f:
        f.write(f"Seed: {seed}\n")
        f.write(f"Peak Memory (GB): {peak_mem:.2f}\n")
        f.write(f"Results (EM / F1): {results['exact_match']:.2f} / {results['f1']:.2f}\n")

    return results

#### Seed 42

In [ ]:
# Run the experiment with a specific seed
results = run_lora_experiment(seed=42)
print(results)

#### Seed 1234

In [ ]:
# Run the experiment with a specific seed
results = run_lora_experiment(seed=1234)
print(results)

#### Seed 2023

In [ ]:
# Run the experiment with a specific seed
results = run_lora_experiment(seed=2023)
print(results)

#### Seed 2024

In [ ]:
# Run the experiment with a specific seed
results = run_lora_experiment(seed=2024)
print(results)

#### Seed 2025

In [ ]:
# Run the experiment with a specific seed
results = run_lora_experiment(seed=2025)
print(results)

### Training BERT with QLoRA for Question Answering

#### Main Experiment Function

In [ ]:
# === qa_qlora_squad.py ===
import os
import evaluate
import numpy as np
import torch
import gc

from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
    pipeline,
    set_seed,
    BitsAndBuytesConfig,
)
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training
)


def run_qlora_experiment(
    seed: int,
    base_model: str = "bert-base-uncased",
    base_dataset: str = "squad",
    output_dir: str = "./results",
    logging_dir: str = "./logs",
    num_epochs: int = 3,
    batch_size: int = 32,
    learning_rate: float = 5e-5,
    device: int = 0,  # GPU device index, -1 for CPU
):
    """
    Run a full train + evaluate pipeline with LoRA on a QA task for a given seed.

    Parameters:
    - seed: Random seed for reproducibility.
    - base_model: Hugging Face model identifier.
    - base_dataset: Hugging Face dataset identifier.
    - output_dir: Root directory for saving model & logs.
    - logging_dir: Directory for training logs.
    - num_epochs: Number of training epochs.
    - batch_size: Training batch size.
    - learning_rate: Initial learning rate.
    - device: GPU device index or -1 for CPU.
    """
    # Set seed
    set_seed(seed)

    # Prepare tokenizer
    tokenizer = BertTokenizerFast.from_pretrained(base_model)

    # Reset GPU memory stats if using GPU
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()  # Reset peak memory stats if using GPU
    if device >= 0:
        torch.cuda.set_device(device)
    else:
        device = -1
    
    # Preprocessing function
    def preprocess_fn(ex):
        tok = tokenizer(
            ex["question"], ex["context"],
            truncation="only_second",
            max_length=384,
            stride=128,
            return_overflowing_tokens=False,
            return_offsets_mapping=True,
            padding="max_length",
        )
        offsets = tok.pop("offset_mapping")
        start_char = ex["answers"]["answer_start"][0]
        end_char = start_char + len(ex["answers"]["text"][0])
        start_idx = end_idx = 0
        for i, (s, e) in enumerate(offsets):
            if s <= start_char < e:
                start_idx = i
            if s < end_char <= e:
                end_idx = i
                break
        tok["start_positions"] = start_idx
        tok["end_positions"] = end_idx
        return tok

    # Load and preprocess dataset
    raw = load_dataset(base_dataset)
    train_ds = raw["train"].map(preprocess_fn, batched=False)
    val_ds = raw["validation"].map(preprocess_fn, batched=False)
    train_ds.set_format(type="torch", columns=["input_ids","attention_mask","start_positions","end_positions"])
    val_ds.set_format(type="torch", columns=["input_ids","attention_mask","start_positions","end_positions"])

    # Setup Config for BitsAndBytesConfig to make the model quantizable
    bnb_config = BitsAndBuytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    # Prepare model for k-bit training
    model = BertForQuestionAnswering.from_pretrained(base_model, quantization_config=bnb_config, device_map="auto")
    model = prepare_model_for_kbit_training(model)

    # Model + LoRA setup
    lora_cfg = LoraConfig(
        r=8, lora_alpha=16,
        target_modules=["query","value"],
        lora_dropout=0.1,
        bias="none",
        task_type="QUESTION_ANS",
        use_rslora=False,
    )
    
    model = get_peft_model(model, lora_cfg)

    # Training arguments
    train_args = TrainingArguments(
        output_dir=os.path.join(output_dir, f"seed_{seed}"),
        per_device_train_batch_size=batch_size,
        num_train_epochs=num_epochs,
        learning_rate=learning_rate,
        fp16=torch.cuda.is_available(),
        logging_dir=os.path.join(logging_dir, f"seed_{seed}"),
        logging_strategy="steps",
        logging_steps=500,
        save_strategy="epoch",
        eval_strategy="epoch",
        report_to="none",
        label_names=["start_positions", "end_positions"],
        gradient_checkpointing=True,  # Enable gradient checkpointing for memory efficiency
        gradient_checkpointing_kwargs={"use_reentrant": False},  # Use non-reentrant gradient checkpointing
        max_grad_norm=None,  # Gradient clipping to avoid exploding gradients
    )
    trainer = Trainer(
        model=model,
        args=train_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=default_data_collator,
    )

    # Training
    print(f"Training with seed {seed}...")
    trainer.train()

    # Check peak memory
    peak_mem = None
    if torch.cuda.is_available():
        peak_mem = torch.cuda.max_memory_allocated() / (1024**3)  # in GB
        print(f"Peak CUDA memory (GB): {peak_mem:.2f}")

    # Save model and tokenizer
    trainer.save_model(os.path.join(output_dir, f"lora_squad_seed_{seed}"))
    tokenizer.save_pretrained(os.path.join(output_dir, f"lora_squad_seed_{seed}"))

    # Evaluate
    print("Evaluating model...")
    # Load LoRA-wrapped model for inference
    base = BertForQuestionAnswering.from_pretrained(base_model)
    infer_model = PeftModel.from_pretrained(base, os.path.join(output_dir, f"seed_{seed}"))
    qa_pipe = pipeline("question-answering", model=infer_model, tokenizer=tokenizer, device=device)
    dataset = load_dataset(base_dataset, split="validation")
    predictions, references = [], []
    for item in dataset:
        out = qa_pipe(question=item["question"], context=item["context"])
        predictions.append({"id": item["id"], "prediction_text": out.get("answer", "")})
        references.append({"id": item["id"], "answers": item["answers"]})
    metric = evaluate.load(base_dataset)
    results = metric.compute(predictions=predictions, references=references)
    print(f"Results (EM / F1): {results['exact_match']:.2f} / {results['f1']:.2f}")

    # Cleanup
    torch.cuda.empty_cache()
    del model, infer_model, trainer
    gc.collect()

    # save to text file
    with open(os.path.join(output_dir, f"results_seed_{seed}.txt"), "w") as f:
        f.write(f"Seed: {seed}\n")
        f.write(f"Peak Memory (GB): {peak_mem:.2f}\n")
        f.write(f"Results (EM / F1): {results['exact_match']:.2f} / {results['f1']:.2f}\n")

    return results

#### Seed 42

In [ ]:
# Use the function to run experiments with different seeds
results = run_qlora_experiment(seed=42)
print(results)

#### Seed 1234


In [ ]:
# Use the function to run experiments with different seeds
results = run_qlora_experiment(seed=1234)
print(results)

#### Seed 2023

In [ ]:
# Use the function to run experiments with different seeds
results = run_qlora_experiment(seed=2023)
print(results)

#### Seed 2024

In [ ]:
# Use the function to run experiments with different seeds
results = run_qlora_experiment(seed=2024)
print(results)

#### Seed 2025

In [ ]:
# Use the function to run experiments with different seeds
results = run_qlora_experiment(seed=2025)
print(results)

### Training BERT with RoRA for Question Answering

#### Main Experiment Function

In [ ]:
# === qa_lora_squad.py ===
import os
import evaluate
import numpy as np
import torch
import gc

from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
    pipeline,
    set_seed,
)
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
)


def run_rora_experiment(
    seed: int,
    base_model: str = "bert-base-uncased",
    base_dataset: str = "squad",
    output_dir: str = "./results",
    logging_dir: str = "./logs",
    num_epochs: int = 3,
    batch_size: int = 32,
    learning_rate: float = 5e-5,
    device: int = 0,  # GPU device index, -1 for CPU
):
    """
    Run a full train + evaluate pipeline with LoRA on a QA task for a given seed.

    Parameters:
    - seed: Random seed for reproducibility.
    - base_model: Hugging Face model identifier.
    - base_dataset: Hugging Face dataset identifier.
    - output_dir: Root directory for saving model & logs.
    - logging_dir: Directory for training logs.
    - num_epochs: Number of training epochs.
    - batch_size: Training batch size.
    - learning_rate: Initial learning rate.
    - device: GPU device index or -1 for CPU.
    """
    # Set seed
    set_seed(seed)

    # Prepare tokenizer
    tokenizer = BertTokenizerFast.from_pretrained(base_model)

    # Reset GPU memory stats if using GPU
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()  # Reset peak memory stats if using GPU
    if device >= 0:
        torch.cuda.set_device(device)
    else:
        device = -1

    # Preprocessing function
    def preprocess_fn(ex):
        tok = tokenizer(
            ex["question"], ex["context"],
            truncation="only_second",
            max_length=384,
            stride=128,
            return_overflowing_tokens=False,
            return_offsets_mapping=True,
            padding="max_length",
        )
        offsets = tok.pop("offset_mapping")
        start_char = ex["answers"]["answer_start"][0]
        end_char = start_char + len(ex["answers"]["text"][0])
        start_idx = end_idx = 0
        for i, (s, e) in enumerate(offsets):
            if s <= start_char < e:
                start_idx = i
            if s < end_char <= e:
                end_idx = i
                break
        tok["start_positions"] = start_idx
        tok["end_positions"] = end_idx
        return tok

    # Load and preprocess dataset
    raw = load_dataset(base_dataset)
    train_ds = raw["train"].map(preprocess_fn, batched=False)
    val_ds = raw["validation"].map(preprocess_fn, batched=False)
    train_ds.set_format(type="torch", columns=["input_ids","attention_mask","start_positions","end_positions"])
    val_ds.set_format(type="torch", columns=["input_ids","attention_mask","start_positions","end_positions"])

    # Model + LoRA setup
    model = BertForQuestionAnswering.from_pretrained(base_model)
    lora_cfg = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["query","value"],
        lora_dropout=0.1,
        bias="none",
        task_type="QUESTION_ANS",
        use_rslora=False,
    )
    model = get_peft_model(model, lora_cfg)

    # Training arguments
    train_args = TrainingArguments(
        output_dir=os.path.join(output_dir, f"seed_{seed}"),
        per_device_train_batch_size=batch_size,
        num_train_epochs=num_epochs,
        learning_rate=learning_rate,
        fp16=torch.cuda.is_available(),
        logging_dir=os.path.join(logging_dir, f"seed_{seed}"),
        logging_strategy="steps",
        logging_steps=500,
        save_strategy="epoch",
        eval_strategy="epoch",
        report_to="none",
        label_names=["start_positions", "end_positions"],
    )
    trainer = Trainer(
        model=model,
        args=train_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=default_data_collator,
    )

    # Training
    print(f"Training with seed {seed}...")
    trainer.train()

    # Check peak memory
    peak_mem = None
    if torch.cuda.is_available():
        peak_mem = torch.cuda.max_memory_allocated() / (1024**3)  # in GB
        print(f"Peak CUDA memory (GB): {peak_mem:.2f}")

    # Save model and tokenizer
    trainer.save_model(os.path.join(output_dir, f"lora_squad_seed_{seed}"))
    tokenizer.save_pretrained(os.path.join(output_dir, f"lora_squad_seed_{seed}"))

    # Evaluate
    print("Evaluating model...")
    # Load LoRA-wrapped model for inference
    base = BertForQuestionAnswering.from_pretrained(base_model)
    infer_model = PeftModel.from_pretrained(base, os.path.join(output_dir, f"seed_{seed}"))
    qa_pipe = pipeline("question-answering", model=infer_model, tokenizer=tokenizer, device=device)
    dataset = load_dataset(base_dataset, split="validation")
    predictions, references = [], []
    for item in dataset:
        out = qa_pipe(question=item["question"], context=item["context"])
        predictions.append({"id": item["id"], "prediction_text": out.get("answer", "")})
        references.append({"id": item["id"], "answers": item["answers"]})
    metric = evaluate.load(base_dataset)
    results = metric.compute(predictions=predictions, references=references)
    print(f"Results (EM / F1): {results['exact_match']:.2f} / {results['f1']:.2f}")

    # Cleanup
    torch.cuda.empty_cache()
    del model, infer_model, trainer
    gc.collect()

    # print into text file
    with open(os.path.join(output_dir, f"results_seed_{seed}.txt"), "w") as f:
        f.write(f"Seed: {seed}\n")
        f.write(f"Peak Memory (GB): {peak_mem:.2f}\n")
        f.write(f"Results (EM / F1): {results['exact_match']:.2f} / {results['f1']:.2f}\n")

    return results

#### Seed 42

In [ ]:
# Run the experiment with a specific seed
results = run_rora_experiment(seed=42)
print(results)

#### Seed 1234

In [ ]:
# Run the experiment with a specific seed
results = run_rora_experiment(seed=1234)
print(results)

#### Seed 2023

In [ ]:
# Run the experiment with a specific seed
results = run_rora_experiment(seed=2023)
print(results)

#### Seed 2024

In [ ]:
# Run the experiment with a specific seed
results = run_rora_experiment(seed=2024)
print(results)

#### Seed 2025

In [ ]:
# Run the experiment with a specific seed
results = run_rora_experiment(seed=2025)
print(results)